# Capstone Part 3: DPO Alignment

Direct Preference Optimization (DPO) aligns the model with human preferences **without**
training a separate reward model. Instead, DPO directly optimizes the policy using
preference pairs (chosen vs rejected responses).

**Key insight:** DPO reparameterizes the RLHF objective so that the optimal policy can be
extracted in closed form from the reward function, yielding a simple classification-like loss.

$$\mathcal{L}_{\text{DPO}} = -\mathbb{E}\left[\log \sigma\left(\beta \log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right)\right]$$

where $y_w$ is the preferred ("chosen") response and $y_l$ is the rejected response.

By the end of this notebook you will have:
- An understanding of DPO theory and implementation
- A DPO-aligned model checkpoint
- Visualizations of reward margins and preference accuracy

## 1. Environment Setup

In [ ]:
!pip install -q torch datasets tiktoken matplotlib tqdm

In [ ]:
import math
import time
import copy
import json
from pathlib import Path
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import tiktoken
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Load Model Architecture and SFT Checkpoint

In [ ]:
# ---- Model definition (same as previous notebooks) ----

@dataclass
class ModelConfig:
    vocab_size: int = 50257
    max_seq_len: int = 512
    d_model: int = 256
    n_layers: int = 8
    n_heads: int = 8
    n_kv_heads: int = 2
    d_ff: int = 688
    dropout: float = 0.1
    rope_theta: float = 10000.0


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return (x * (x.float().pow(2).mean(-1, keepdim=True) + self.eps).rsqrt()).type_as(x) * self.weight


def precompute_rope_frequencies(dim, max_seq_len, theta=10000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
    angles = torch.outer(torch.arange(max_seq_len).float(), freqs)
    return torch.polar(torch.ones_like(angles), angles)


def apply_rope(x, freqs_cis):
    b, s, n, d = x.shape
    x_c = torch.view_as_complex(x.float().reshape(b, s, n, -1, 2))
    fc = freqs_cis[:s].unsqueeze(0).unsqueeze(2)
    return torch.view_as_real(x_c * fc).reshape(b, s, n, d).type_as(x)


class GroupedQueryAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_heads, self.n_kv_heads = config.n_heads, config.n_kv_heads
        self.head_dim = config.d_model // config.n_heads
        self.n_rep = self.n_heads // self.n_kv_heads
        self.wq = nn.Linear(config.d_model, config.n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(config.d_model, config.n_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(config.d_model, config.n_kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(config.n_heads * self.head_dim, config.d_model, bias=False)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

    def _repeat_kv(self, x):
        if self.n_rep == 1: return x
        b, s, n, d = x.shape
        return x[:,:,:,None,:].expand(b,s,n,self.n_rep,d).reshape(b,s,self.n_heads,d)

    def forward(self, x, freqs_cis, mask=None):
        b, s, _ = x.shape
        q = self.wq(x).reshape(b,s,self.n_heads,self.head_dim)
        k = self.wk(x).reshape(b,s,self.n_kv_heads,self.head_dim)
        v = self.wv(x).reshape(b,s,self.n_kv_heads,self.head_dim)
        q, k = apply_rope(q, freqs_cis), apply_rope(k, freqs_cis)
        k, v = self._repeat_kv(k), self._repeat_kv(v)
        q, k, v = q.transpose(1,2), k.transpose(1,2), v.transpose(1,2)
        attn = torch.matmul(q, k.transpose(-2,-1)) / math.sqrt(self.head_dim)
        if mask is not None: attn = attn.masked_fill(mask == 0, float("-inf"))
        out = torch.matmul(self.attn_dropout(F.softmax(attn, dim=-1)), v)
        return self.resid_dropout(self.wo(out.transpose(1,2).reshape(b,s,-1)))


class SwiGLU(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.w1 = nn.Linear(config.d_model, config.d_ff, bias=False)
        self.w2 = nn.Linear(config.d_ff, config.d_model, bias=False)
        self.w3 = nn.Linear(config.d_model, config.d_ff, bias=False)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        return self.dropout(self.w2(F.silu(self.w1(x)) * self.w3(x)))


class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attention_norm = RMSNorm(config.d_model)
        self.attention = GroupedQueryAttention(config)
        self.ffn_norm = RMSNorm(config.d_model)
        self.ffn = SwiGLU(config)

    def forward(self, x, freqs_cis, mask=None):
        x = x + self.attention(self.attention_norm(x), freqs_cis, mask)
        return x + self.ffn(self.ffn_norm(x))


class MiniLLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.tok_emb = nn.Embedding(config.vocab_size, config.d_model)
        self.dropout = nn.Dropout(config.dropout)
        self.layers = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layers)])
        self.norm = RMSNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
        head_dim = config.d_model // config.n_heads
        self.register_buffer("freqs_cis", precompute_rope_frequencies(head_dim, config.max_seq_len, config.rope_theta), persistent=False)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None: torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, targets=None):
        b, s = input_ids.shape
        mask = torch.tril(torch.ones(s, s, device=input_ids.device)).unsqueeze(0).unsqueeze(0)
        x = self.dropout(self.tok_emb(input_ids))
        for layer in self.layers:
            x = layer(x, self.freqs_cis, mask)
        logits = self.lm_head(self.norm(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        return logits, loss

    def get_log_probs(self, input_ids, labels):
        """Compute per-token log probabilities for DPO."""
        logits, _ = self(input_ids)
        log_probs = F.log_softmax(logits, dim=-1)
        # Gather log probs for target tokens
        per_token_log_probs = torch.gather(
            log_probs[:, :-1, :], 2, labels[:, 1:].unsqueeze(-1)
        ).squeeze(-1)
        # Mask out padding/instruction tokens
        mask = (labels[:, 1:] != -1).float()
        return (per_token_log_probs * mask).sum(-1)  # Sum over sequence

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=100, temperature=0.8, top_k=50):
        for _ in range(max_new_tokens):
            idx = input_ids[:, -self.config.max_seq_len:]
            logits, _ = self(idx)
            logits = logits[:, -1, :] / temperature
            if top_k > 0:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            input_ids = torch.cat([input_ids, torch.multinomial(F.softmax(logits, dim=-1), 1)], dim=1)
        return input_ids


print("Model architecture defined.")

In [ ]:
# Load SFT checkpoint
save_dir = Path("checkpoints")
sft_checkpoint = torch.load(save_dir / "best_sft.pt", map_location=device, weights_only=False)

config = sft_checkpoint["config"]
tokenizer = tiktoken.get_encoding("gpt2")

# Policy model (will be trained with DPO)
policy_model = MiniLLM(config).to(device)
policy_model.load_state_dict(sft_checkpoint["model_state_dict"])

# Reference model (frozen copy of SFT model)
ref_model = MiniLLM(config).to(device)
ref_model.load_state_dict(sft_checkpoint["model_state_dict"])
ref_model.eval()
for param in ref_model.parameters():
    param.requires_grad = False

print(f"Loaded SFT checkpoint (step {sft_checkpoint['step']})")
print(f"Policy model parameters: {sum(p.numel() for p in policy_model.parameters()):,}")
print(f"Reference model frozen: {not any(p.requires_grad for p in ref_model.parameters())}")

## 3. Understanding DPO

DPO works by comparing the policy model's log-probabilities on **chosen** vs **rejected**
responses, relative to a frozen **reference** model.

The implicit reward for a response $y$ given prompt $x$ is:

$$r(x, y) = \beta \log \frac{\pi_\theta(y|x)}{\pi_{\text{ref}}(y|x)}$$

DPO maximizes the margin between chosen and rejected rewards:

$$\mathcal{L}_{\text{DPO}} = -\log \sigma(r(x, y_w) - r(x, y_l))$$

where $\sigma$ is the sigmoid function. This is essentially a binary classification loss:
"Does the model assign higher reward to the chosen response?"

The $\beta$ parameter controls how far the policy can deviate from the reference:
- **Low $\beta$** (0.01-0.05): Policy stays close to reference
- **High $\beta$** (0.5-1.0): Policy can diverge significantly

In [ ]:
# Visualize the DPO loss landscape
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# DPO loss as a function of reward margin
margins = torch.linspace(-5, 5, 200)
for beta in [0.05, 0.1, 0.2, 0.5]:
    loss = -F.logsigmoid(beta * margins)
    axes[0].plot(margins.numpy(), loss.numpy(), label=f'beta={beta}')

axes[0].set_xlabel('Log-ratio margin (chosen - rejected)')
axes[0].set_ylabel('DPO Loss')
axes[0].set_title('DPO Loss vs Reward Margin')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].axvline(x=0, color='gray', linestyle='--', alpha=0.5)

# Gradient magnitude
for beta in [0.05, 0.1, 0.2, 0.5]:
    grad = -beta * torch.sigmoid(-beta * margins)
    axes[1].plot(margins.numpy(), grad.abs().numpy(), label=f'beta={beta}')

axes[1].set_xlabel('Log-ratio margin')
axes[1].set_ylabel('|Gradient|')
axes[1].set_title('DPO Gradient Magnitude')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('DPO Loss Landscape', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Key observations:")
print("- Loss is high when the model assigns higher probability to the rejected response")
print("- Loss approaches 0 as the margin increases (model strongly prefers chosen)")
print("- Gradient is strongest near margin=0 (most informative region)")

## 4. Preference Dataset

We create preference pairs where each example has:
- A **prompt** (instruction)
- A **chosen** response (preferred by humans)
- A **rejected** response (less preferred)

For our simple model, preferences encode basic quality signals:
- Complete vs incomplete stories
- On-topic vs off-topic responses
- Coherent vs incoherent text

In [ ]:
INSTRUCTION_PREFIX = "<|instruction|> "
RESPONSE_PREFIX = " <|response|> "
END_TOKEN = " <|end|>"

# Hand-crafted preference pairs
PREFERENCE_DATA = [
    {
        "prompt": "Write a short story about a cat.",
        "chosen": "Once upon a time, there was a fluffy cat named Luna. Luna loved to chase butterflies in the garden. One day, she found a baby bird that had fallen from its nest. Instead of chasing it, Luna gently nudged it back toward the tree. The mama bird sang a happy song to thank Luna. From that day on, Luna and the birds were friends.",
        "rejected": "Cat. The cat was there. It did things. The end."
    },
    {
        "prompt": "Tell me a story about friendship.",
        "chosen": "Max and Lily were neighbors who became best friends. They played together every day after school. One rainy day, Max was sad because he lost his favorite book. Lily helped him search everywhere. They found it under the couch! Max gave Lily a big hug. They learned that true friends always help each other.",
        "rejected": "Friendship is when people are friends. Some people have friends and some do not. Friends are nice to have. You can play with friends."
    },
    {
        "prompt": "Write about a sunny day.",
        "chosen": "The sun shone bright over the little town. Children played in the park, laughing and running. An ice cream truck drove by playing its cheerful song. Emma got a strawberry cone and sat under the big oak tree. The warm breeze felt wonderful. It was the perfect summer day.",
        "rejected": "It was sunny. The sun was in the sky. It was bright. The day was nice. Very sunny and warm."
    },
    {
        "prompt": "Tell a story about a dog learning a new trick.",
        "chosen": "Buddy was a playful golden retriever who wanted to learn to shake hands. Every day, his owner Sarah would hold a treat and say shake. At first, Buddy just wagged his tail. After many tries, he lifted his paw a little. Sarah cheered! By the end of the week, Buddy could shake hands perfectly. He was so proud of himself.",
        "rejected": "The dog liked food. Food was good. The dog ate the food and then more food. Dogs eat a lot of food."
    },
    {
        "prompt": "Write a bedtime story.",
        "chosen": "As the moon rose high in the sky, little Bear yawned and rubbed his eyes. Mama Bear tucked him into his cozy bed. She told him about the stars, how each one was a wish waiting to come true. Bear closed his eyes and made a wish for a sunny day tomorrow. He drifted off to sleep with a smile on his face. Good night, little Bear.",
        "rejected": "Go to sleep. It is night time. Close your eyes. Sleep now. The story is over."
    },
    {
        "prompt": "Write about a trip to the beach.",
        "chosen": "Mia and her family drove to the beach on a warm Saturday morning. The sand was soft between her toes. She built a magnificent sandcastle with towers and a moat. Her brother found colorful shells along the shore. They splashed in the cool waves until the sun began to set, painting the sky orange and pink.",
        "rejected": "The beach had sand and water. There were waves. It was wet. Sand is on the beach. Water is also on the beach."
    },
    {
        "prompt": "Tell a story about being kind.",
        "chosen": "Sophie noticed the new girl at school sitting alone at lunch. She walked over with her tray and asked if she could sit down. The new girl smiled and said yes. They talked about their favorite books and discovered they both loved adventure stories. By the end of lunch, they had made plans to read together. One small act of kindness made two people happy.",
        "rejected": "Be kind to people. Kind is good. You should be kind because it is nice. Being kind is important."
    },
    {
        "prompt": "Write about a garden.",
        "chosen": "Grandma's garden was a magical place. Red roses climbed up the old stone wall. Bees buzzed happily from flower to flower. In one corner, tomatoes grew fat and red on the vine. Grandma taught me how to water the plants gently. I learned that with patience and care, beautiful things can grow.",
        "rejected": "The garden had plants. Plants need water. The garden was green. There were flowers in the garden. Gardens have dirt."
    },
    {
        "prompt": "Write a story about sharing.",
        "chosen": "Tom had two cookies and his friend Sam had none. Tom saw that Sam looked sad. Without being asked, Tom broke one cookie in half and gave the bigger piece to Sam. Sam's face lit up with a huge smile. They sat together and enjoyed their cookies. Tom felt warm inside because making his friend happy made him happy too.",
        "rejected": "Sharing is when you give things to other people. You should share because your parents tell you to share."
    },
    {
        "prompt": "Tell a story about a rainy day.",
        "chosen": "Rain pattered against the window as Jake watched from inside. He was bored until Mom suggested they build a blanket fort in the living room. They used every pillow and blanket they could find. Inside the fort, they read stories by flashlight and drank hot cocoa. Jake decided that rainy days might just be his new favorite.",
        "rejected": "It rained. Rain comes from clouds. Clouds are in the sky. When it rains you get wet if you go outside."
    },
    {
        "prompt": "Write about a birthday.",
        "chosen": "Today was Emma's seventh birthday! She woke up to the smell of pancakes shaped like stars. Her friends came over for a party in the backyard. They played musical chairs and pin the tail on the donkey. The cake was chocolate with rainbow sprinkles. Emma made a wish and blew out all seven candles in one breath. It was the happiest day of her life.",
        "rejected": "Birthday. Happy birthday. Cake and presents. Birthday party. Candles on cake."
    },
    {
        "prompt": "Write about learning something new.",
        "chosen": "Mia wanted to learn to paint. At first, her paintings looked nothing like what she imagined. The trees were blurry and the sky was too dark. But her art teacher encouraged her to keep practicing. Each week, her paintings got a little better. After three months, Mia painted a beautiful sunset that made her whole family smile. She learned that practice turns mistakes into masterpieces.",
        "rejected": "Learning is good. You learn things at school. Learning new things is important for everyone."
    },
]

print(f"Preference pairs: {len(PREFERENCE_DATA)}")
print(f"\nSample pair:")
print(f"  Prompt: {PREFERENCE_DATA[0]['prompt']}")
print(f"  Chosen: {PREFERENCE_DATA[0]['chosen'][:100]}...")
print(f"  Rejected: {PREFERENCE_DATA[0]['rejected'][:100]}...")

## 5. DPO Dataset

Each example produces two tokenized sequences (chosen and rejected) with the same prompt.
We also create labels that mask the prompt tokens.

In [ ]:
class DPODataset(Dataset):
    """Dataset for DPO training with chosen/rejected pairs."""

    def __init__(self, preference_data, tokenizer, max_len=512):
        self.samples = []
        self.tokenizer = tokenizer

        for pair in preference_data:
            prompt_text = INSTRUCTION_PREFIX + pair["prompt"] + RESPONSE_PREFIX
            chosen_text = prompt_text + pair["chosen"] + END_TOKEN
            rejected_text = prompt_text + pair["rejected"] + END_TOKEN

            prompt_tokens = tokenizer.encode(prompt_text)
            chosen_tokens = tokenizer.encode(chosen_text)[:max_len]
            rejected_tokens = tokenizer.encode(rejected_text)[:max_len]

            def make_padded(tokens, prompt_len):
                input_ids = tokens[:-1]
                labels = [-1] * (prompt_len - 1) + tokens[prompt_len - 1:-1]
                labels = labels[:len(input_ids)]
                pad_len = max_len - len(input_ids) - 1
                if pad_len > 0:
                    input_ids = input_ids + [0] * pad_len
                    labels = labels + [-1] * pad_len
                return (
                    torch.tensor(input_ids[:max_len-1], dtype=torch.long),
                    torch.tensor(labels[:max_len-1], dtype=torch.long),
                )

            chosen_ids, chosen_labels = make_padded(chosen_tokens, len(prompt_tokens))
            rejected_ids, rejected_labels = make_padded(rejected_tokens, len(prompt_tokens))

            self.samples.append({
                "chosen_ids": chosen_ids,
                "chosen_labels": chosen_labels,
                "rejected_ids": rejected_ids,
                "rejected_labels": rejected_labels,
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        return s["chosen_ids"], s["chosen_labels"], s["rejected_ids"], s["rejected_labels"]


dpo_dataset = DPODataset(PREFERENCE_DATA, tokenizer, max_len=config.max_seq_len)
dpo_loader = DataLoader(dpo_dataset, batch_size=4, shuffle=True, num_workers=0)

print(f"DPO dataset size: {len(dpo_dataset)}")
print(f"Batches per epoch: {len(dpo_loader)}")

# Inspect a sample
c_ids, c_labels, r_ids, r_labels = dpo_dataset[0]
print(f"\nChosen input length: {c_ids.shape[0]}")
print(f"Chosen active tokens: {(c_labels != -1).sum().item()}")
print(f"Rejected input length: {r_ids.shape[0]}")
print(f"Rejected active tokens: {(r_labels != -1).sum().item()}")

## 6. DPO Loss Implementation

The DPO loss computes:
1. Log-probabilities of chosen/rejected under the **policy** model
2. Log-probabilities of chosen/rejected under the frozen **reference** model
3. The implicit reward margin
4. Binary cross-entropy loss on the margin

In [ ]:
def compute_log_probs(model, input_ids, labels):
    """Compute sum of log-probabilities for active (non-masked) tokens."""
    logits, _ = model(input_ids)
    # Shift logits and labels for next-token prediction
    shift_logits = logits[:, :-1, :]
    shift_labels = labels[:, 1:]

    log_probs = F.log_softmax(shift_logits, dim=-1)
    # Gather log probs for target tokens
    per_token_log_probs = torch.gather(
        log_probs, 2, shift_labels.clamp(min=0).unsqueeze(-1)
    ).squeeze(-1)

    # Mask: only compute on response tokens (labels != -1)
    mask = (shift_labels != -1).float()
    # Average log prob per active token (prevents length bias)
    return (per_token_log_probs * mask).sum(-1) / mask.sum(-1).clamp(min=1)


def dpo_loss(policy_model, ref_model, chosen_ids, chosen_labels,
             rejected_ids, rejected_labels, beta=0.1):
    """Compute the DPO loss.

    Returns:
        loss: scalar DPO loss
        metrics: dict with chosen/rejected rewards and accuracy
    """
    # Policy log-probs
    policy_chosen_logps = compute_log_probs(policy_model, chosen_ids, chosen_labels)
    policy_rejected_logps = compute_log_probs(policy_model, rejected_ids, rejected_labels)

    # Reference log-probs (no gradient)
    with torch.no_grad():
        ref_chosen_logps = compute_log_probs(ref_model, chosen_ids, chosen_labels)
        ref_rejected_logps = compute_log_probs(ref_model, rejected_ids, rejected_labels)

    # Implicit rewards
    chosen_rewards = beta * (policy_chosen_logps - ref_chosen_logps)
    rejected_rewards = beta * (policy_rejected_logps - ref_rejected_logps)

    # DPO loss = -log(sigmoid(reward_chosen - reward_rejected))
    reward_margin = chosen_rewards - rejected_rewards
    loss = -F.logsigmoid(reward_margin).mean()

    # Metrics
    with torch.no_grad():
        accuracy = (reward_margin > 0).float().mean().item()
        metrics = {
            "loss": loss.item(),
            "chosen_reward": chosen_rewards.mean().item(),
            "rejected_reward": rejected_rewards.mean().item(),
            "reward_margin": reward_margin.mean().item(),
            "accuracy": accuracy,
        }

    return loss, metrics


# Test with a single batch
policy_model.train()
c_ids, c_labels, r_ids, r_labels = next(iter(dpo_loader))
c_ids, c_labels = c_ids.to(device), c_labels.to(device)
r_ids, r_labels = r_ids.to(device), r_labels.to(device)

loss, metrics = dpo_loss(policy_model, ref_model, c_ids, c_labels, r_ids, r_labels, beta=0.1)
print("Initial DPO metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")
print(f"\nNote: Initial accuracy ~50% means model has no preference yet (expected).")

## 7. Pre-DPO Baseline

Record how the SFT model responds before DPO training.

In [ ]:
def generate_response(model, tokenizer, instruction, device, max_tokens=100):
    prompt = INSTRUCTION_PREFIX + instruction + RESPONSE_PREFIX
    input_ids = torch.tensor([tokenizer.encode(prompt)], device=device)
    output_ids = model.generate(input_ids, max_new_tokens=max_tokens, temperature=0.7, top_k=40)
    full_text = tokenizer.decode(output_ids[0].tolist())
    if "<|response|>" in full_text:
        response = full_text.split("<|response|>")[-1]
        if "<|end|>" in response:
            response = response.split("<|end|>")[0]
        return response.strip()
    return full_text[len(prompt):].strip()


eval_instructions = [
    "Write a short story about a rabbit.",
    "Tell a story about helping someone.",
    "Write about a winter day.",
]

print("=" * 70)
print("PRE-DPO BASELINE (SFT model)")
print("=" * 70)

policy_model.eval()
pre_dpo_responses = {}
for instr in eval_instructions:
    resp = generate_response(policy_model, tokenizer, instr, device)
    pre_dpo_responses[instr] = resp
    print(f"\nInstruction: {instr}")
    print(f"Response: {resp[:250]}")
    print("-" * 70)

## 8. DPO Training Configuration

In [ ]:
@dataclass
class DPOTrainConfig:
    num_epochs: int = 10        # More epochs since dataset is small
    learning_rate: float = 1e-5  # Very low LR for alignment
    beta: float = 0.1            # KL penalty coefficient
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    log_interval: int = 5

dpo_config = DPOTrainConfig()

optimizer = torch.optim.AdamW(
    policy_model.parameters(),
    lr=dpo_config.learning_rate,
    weight_decay=dpo_config.weight_decay,
    betas=(0.9, 0.95),
)

total_steps = len(dpo_loader) * dpo_config.num_epochs
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, total_steps, eta_min=1e-6)

print(f"DPO training config:")
print(f"  Epochs: {dpo_config.num_epochs}")
print(f"  Learning rate: {dpo_config.learning_rate}")
print(f"  Beta: {dpo_config.beta}")
print(f"  Total steps: {total_steps}")

## 9. DPO Training Loop

In [ ]:
# Tracking
dpo_losses = []
dpo_accuracies = []
chosen_rewards_history = []
rejected_rewards_history = []
margins_history = []
global_step = 0
best_accuracy = 0.0

print(f"Starting DPO training for {dpo_config.num_epochs} epochs")
print("=" * 70)

policy_model.train()
for epoch in range(dpo_config.num_epochs):
    epoch_metrics = {"loss": 0, "accuracy": 0, "reward_margin": 0}
    n_batches = 0

    for c_ids, c_labels, r_ids, r_labels in dpo_loader:
        c_ids, c_labels = c_ids.to(device), c_labels.to(device)
        r_ids, r_labels = r_ids.to(device), r_labels.to(device)

        loss, metrics = dpo_loss(
            policy_model, ref_model,
            c_ids, c_labels, r_ids, r_labels,
            beta=dpo_config.beta
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy_model.parameters(), dpo_config.max_grad_norm)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()

        # Track
        dpo_losses.append(metrics["loss"])
        dpo_accuracies.append(metrics["accuracy"])
        chosen_rewards_history.append(metrics["chosen_reward"])
        rejected_rewards_history.append(metrics["rejected_reward"])
        margins_history.append(metrics["reward_margin"])

        for k in epoch_metrics:
            epoch_metrics[k] += metrics[k]
        n_batches += 1
        global_step += 1

        if global_step % dpo_config.log_interval == 0:
            print(
                f"  Step {global_step}/{total_steps} | "
                f"Loss: {metrics['loss']:.4f} | "
                f"Acc: {metrics['accuracy']:.2f} | "
                f"Margin: {metrics['reward_margin']:.4f}"
            )

    # Epoch summary
    for k in epoch_metrics:
        epoch_metrics[k] /= n_batches

    print(
        f"\nEpoch {epoch+1}/{dpo_config.num_epochs} | "
        f"Loss: {epoch_metrics['loss']:.4f} | "
        f"Acc: {epoch_metrics['accuracy']:.2f} | "
        f"Margin: {epoch_metrics['reward_margin']:.4f}"
    )

    if epoch_metrics["accuracy"] > best_accuracy:
        best_accuracy = epoch_metrics["accuracy"]
        torch.save({
            "model_state_dict": policy_model.state_dict(),
            "config": config,
            "step": global_step,
            "accuracy": best_accuracy,
            "stage": "dpo",
        }, save_dir / "best_dpo.pt")
        print(f"  >> Saved best DPO checkpoint (accuracy={best_accuracy:.2f})")
    print("=" * 70)

# Save final
torch.save({
    "model_state_dict": policy_model.state_dict(),
    "config": config,
    "step": global_step,
    "stage": "dpo",
}, save_dir / "final_dpo.pt")
print("DPO training complete!")

## 10. DPO Training Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# DPO Loss
ax = axes[0, 0]
ax.plot(dpo_losses, alpha=0.5, color='blue')
if len(dpo_losses) > 5:
    smoothed = [sum(dpo_losses[max(0,i-5):i+1])/min(i+1,5) for i in range(len(dpo_losses))]
    ax.plot(smoothed, color='blue', linewidth=2)
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('DPO Loss')
ax.grid(True, alpha=0.3)

# Preference accuracy
ax = axes[0, 1]
ax.plot(dpo_accuracies, 'g.-', alpha=0.7)
ax.axhline(y=0.5, color='red', linestyle='--', label='Random (50%)')
ax.set_xlabel('Step')
ax.set_ylabel('Accuracy')
ax.set_title('Preference Accuracy (chosen > rejected)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.05, 1.05)

# Reward values
ax = axes[1, 0]
ax.plot(chosen_rewards_history, label='Chosen', color='green', alpha=0.7)
ax.plot(rejected_rewards_history, label='Rejected', color='red', alpha=0.7)
ax.set_xlabel('Step')
ax.set_ylabel('Implicit Reward')
ax.set_title('Chosen vs Rejected Rewards')
ax.legend()
ax.grid(True, alpha=0.3)

# Reward margin
ax = axes[1, 1]
ax.plot(margins_history, color='purple', alpha=0.7)
ax.axhline(y=0, color='gray', linestyle='--')
ax.fill_between(range(len(margins_history)), margins_history, 0,
                where=[m > 0 for m in margins_history], alpha=0.3, color='green', label='Correct')
ax.fill_between(range(len(margins_history)), margins_history, 0,
                where=[m <= 0 for m in margins_history], alpha=0.3, color='red', label='Incorrect')
ax.set_xlabel('Step')
ax.set_ylabel('Reward Margin')
ax.set_title('Reward Margin (chosen - rejected)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('DPO Training Dashboard', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(save_dir / 'dpo_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Post-DPO Generation

In [ ]:
policy_model.eval()

print("=" * 70)
print("POST-DPO GENERATION")
print("=" * 70)

post_dpo_responses = {}
for instr in eval_instructions:
    resp = generate_response(policy_model, tokenizer, instr, device)
    post_dpo_responses[instr] = resp
    print(f"\nInstruction: {instr}")
    print(f"Response: {resp[:300]}")
    print("-" * 70)

## 12. Side-by-Side: SFT vs DPO

In [ ]:
print("=" * 80)
print("COMPARISON: SFT vs DPO")
print("=" * 80)

for instr in eval_instructions:
    print(f"\nInstruction: {instr}")
    print(f"\n  [SFT]  {pre_dpo_responses[instr][:200]}")
    print(f"\n  [DPO]  {post_dpo_responses[instr][:200]}")
    print("-" * 80)

## 13. KL Divergence Analysis

An important metric during DPO is how much the policy diverges from the reference.
Too much divergence can lead to reward hacking; too little means no alignment.

In [ ]:
# Measure KL divergence between policy and reference on some prompts
@torch.no_grad()
def compute_kl_divergence(policy_model, ref_model, input_ids):
    """Compute average per-token KL(policy || ref)."""
    policy_logits, _ = policy_model(input_ids)
    ref_logits, _ = ref_model(input_ids)

    policy_log_probs = F.log_softmax(policy_logits, dim=-1)
    ref_log_probs = F.log_softmax(ref_logits, dim=-1)

    # KL(policy || ref) = sum(policy * (log_policy - log_ref))
    kl = F.kl_div(ref_log_probs, policy_log_probs, log_target=True, reduction='none')
    return kl.sum(-1).mean().item()  # Average over batch and sequence


# Compute KL for several prompts
kl_prompts = [
    "Once upon a time",
    "The little dog went",
    "<|instruction|> Write a story <|response|>",
    "<|instruction|> Tell me about <|response|>",
]

policy_model.eval()
kl_values = []
for prompt in kl_prompts:
    ids = torch.tensor([tokenizer.encode(prompt)], device=device)
    kl = compute_kl_divergence(policy_model, ref_model, ids)
    kl_values.append(kl)
    print(f"KL divergence for '{prompt[:40]}...': {kl:.4f}")

print(f"\nAverage KL divergence: {sum(kl_values)/len(kl_values):.4f}")
print("A moderate KL (0.01-0.5) suggests meaningful but controlled alignment.")

## 14. Reward Margin Analysis on Held-Out Pairs

Let's test the model on new preference pairs it hasn't seen during training.

In [ ]:
# New preference pairs for evaluation
eval_pairs = [
    {
        "prompt": "Write about a kitten.",
        "chosen": "A tiny kitten named Daisy curled up in a warm patch of sunlight. She purred softly as she dreamed of chasing leaves in the garden. When she woke up, she stretched and went to find her favorite toy mouse.",
        "rejected": "Kitten small. Very small. Kitten exists.",
    },
    {
        "prompt": "Tell a story about cooking.",
        "chosen": "Grandpa taught me how to make his special soup. We chopped carrots and potatoes together while he told stories about his childhood. The soup simmered on the stove, filling the kitchen with a wonderful smell. When it was ready, the whole family gathered to eat.",
        "rejected": "Cooking involves heat. Food gets hot. You eat it.",
    },
]

eval_dpo = DPODataset(eval_pairs, tokenizer, max_len=config.max_seq_len)
eval_loader = DataLoader(eval_dpo, batch_size=2, shuffle=False)

policy_model.eval()
for c_ids, c_labels, r_ids, r_labels in eval_loader:
    c_ids, c_labels = c_ids.to(device), c_labels.to(device)
    r_ids, r_labels = r_ids.to(device), r_labels.to(device)

    _, metrics = dpo_loss(policy_model, ref_model, c_ids, c_labels, r_ids, r_labels, beta=dpo_config.beta)
    print("Held-out evaluation metrics:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

print("\nAccuracy > 0.5 on held-out data suggests generalization of preferences.")

## 15. Checkpoint Summary

In [ ]:
dpo_ckpt = torch.load(save_dir / "best_dpo.pt", map_location=device, weights_only=False)
print("DPO Checkpoint:")
print(f"  Stage: {dpo_ckpt['stage']}")
print(f"  Step: {dpo_ckpt['step']}")
print(f"  Accuracy: {dpo_ckpt['accuracy']:.2f}")

print("\nAll checkpoints:")
for f in sorted(save_dir.glob("*.pt")):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name}: {size_mb:.1f} MB")

## Summary

In this notebook we:

1. **Implemented** the DPO loss from scratch with detailed explanation
2. **Created** preference pairs encoding quality signals
3. **Trained** DPO with a frozen reference model
4. **Visualized** reward margins, accuracy, and KL divergence
5. **Compared** SFT vs DPO responses side-by-side

The DPO model should now prefer generating detailed, coherent, story-like responses
over terse or low-quality text. In the next notebook we will quantize this model
for efficient inference.

**Checkpoint location:** `checkpoints/best_dpo.pt`